<div style="width: 100%; overflow: hidden;">
    <div style="width: 150px; float: left;"> <img src="data/D4Sci_logo_ball.png" alt="Data For Science, Inc" align="left" border="0"> </div>
    <div style="float: left; margin-left: 10px;"> <h1>GraphRAG</h1>
<h1>Chatbot, Evaluation & Production</h1>
        <p>Bruno Gonçalves<br/>
        <a href="http://www.data4sci.com/">www.data4sci.com</a><br/>
            @bgoncalves, @data4sci</p></div>
</div>

In [1]:
import difflib
import json
import os
import re
from pathlib import Path

import networkx as nx
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import torch
import matplotlib.pyplot as plt 

import watermark

%load_ext watermark
%matplotlib inline

We start by print out the versions of the libraries we're using for future reference

In [2]:
%watermark -n -v -m -g -iv

Python implementation: CPython
Python version       : 3.12.13
IPython version      : 9.15.0

Compiler    : Clang 22.1.3 
OS          : Darwin
Release     : 25.5.0
Machine     : arm64
Processor   : arm
CPU cores   : 16
Architecture: 64bit

Git hash: 9f7b8f641170ae4066def95c2266e8f9ada7bc64

json      : 2.0.9
matplotlib: 3.11.0
networkx  : 3.6.1
re        : 2.2.1
sklearn   : 1.9.0
spacy     : 3.8.14
torch     : 2.12.1
watermark : 2.6.0



Load default figure style

In [3]:
plt.style.use('d4sci.mplstyle')
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

And detect what hardware we're working with

In [4]:
if torch.cuda.is_available():
    DEVICE = "cuda"          # NVIDIA GPU
elif torch.backends.mps.is_available():
    DEVICE = "mps"           # Apple Silicon GPU
else:
    DEVICE = "cpu"
    
print(f"models will run on: {DEVICE}")

models will run on: mps


## Checkpoint system
Every part of this workshop `READS` its inputs from, and `WRITES` its outputs to,
a shared `checkpoints/` directory next to the notebooks. If an expected
checkpoint is missing (you skipped a part, or a stage failed on your machine),
we fall back to built-in data so THIS notebook still runs end to end.

In [5]:
CKPT = Path("checkpoints")
CKPT.mkdir(exist_ok=True)

def save_text(name, text):
    (CKPT / name).write_text(text, encoding="utf-8")
    print(f"[checkpoint] saved  {CKPT / name}  ({len(text):,} chars)")

def load_text(name, fallback=None):
    p = CKPT / name
    if p.exists():
        print(f"[checkpoint] loaded {p}")
        return p.read_text(encoding="utf-8")
    print(f"[checkpoint] {p} NOT FOUND — using built-in fallback data")
    return fallback

def save_jsonl(name, rows):
    with open(CKPT / name, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    print(f"[checkpoint] saved  {CKPT / name}  ({len(rows)} rows)")

def load_jsonl(name, fallback=None):
    p = CKPT / name
    if p.exists():
        rows = [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()]
        print(f"[checkpoint] loaded {p}  ({len(rows)} rows)")
        return rows
    print(f"[checkpoint] {p} NOT FOUND — using built-in fallback data")
    return fallback

In [6]:
rows = load_jsonl("triples.jsonl")
triples = [(r["head"], r["relation"], r["tail"]) for r in rows]

# resolved.txt feeds the EVALUATION baseline (it retrieves over the same text
# the graph was built from — a fair fight).
resolved_text = load_text("resolved.txt")
print(f"\n{len(triples)} triples · {len(resolved_text):,} chars of resolved text")

[checkpoint] loaded checkpoints/triples.jsonl  (2735721 rows)
[checkpoint] loaded checkpoints/resolved.txt

2735721 triples · 634,374,189 chars of resolved text


Rebuild the graph and its traversal helpers (as in Part 3)

In [12]:
G = nx.MultiDiGraph()

for head, rel, tail in triples:
    G.add_edge(head, tail, relation=rel)

def facts_about(entity):
    """All edges touching an entity, both directions, as readable facts."""
    facts = []
    if entity not in G:
        return facts
    for _, tail, d in G.out_edges(entity, data=True):
        facts.append(f"{entity} —{d['relation']}→ {tail}")
    for head, _, d in G.in_edges(entity, data=True):
        facts.append(f"{head} —{d['relation']}→ {entity}")
    return facts

def path_between(a, b):
    """Shortest chain of facts connecting two entities (undirected view)."""
    try:
        nodes = nx.shortest_path(G.to_undirected(), a, b)
    except (nx.NodeNotFound, nx.NetworkXNoPath):
        return None
    hops = []
    for u, v in zip(nodes, nodes[1:]):
        edge = (G.get_edge_data(u, v) or G.get_edge_data(v, u))
        rel = list(edge.values())[0]["relation"]
        hops.append(f"{u} —{rel}— {v}")
    return hops

#hub = max(G.nodes, key=lambda n: G.degree(n))
hub = "Japan"
print(f"Graph rebuilt: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges · hub = {hub}")

Graph rebuilt: 1446345 nodes, 2735721 edges · hub = Japan


## The chatbot

Precompute a lowercase index of node names once.

In [13]:
node_index = {n.lower(): n for n in G.nodes}

def link_entities(question, max_entities=3):
    """Map a question to graph nodes.

    Two passes:
      1. Substring: does any node name appear verbatim in the question?
         (Longest names first, so 'Pierre Curie' beats 'Curie'.)
      2. Fuzzy: for remaining capitalized phrases, find the closest node name —
         catches typos and small variations.
    """
    q = question.lower()
    found = []

    for name_lc in sorted(node_index, key=len, reverse=True):      # pass 1
        if len(name_lc) > 3 and name_lc in q:
            node = node_index[name_lc]
            if node not in found:
                found.append(node)

    if len(found) < max_entities:                                   # pass 2
        candidates = re.findall(r"[A-Z][a-zA-Z]+(?:\s+[A-Z][a-zA-Z]+)*", question)
        for cand in candidates:
            close = difflib.get_close_matches(cand.lower(), node_index.keys(),
                                              n=1, cutoff=0.75)
            if close and node_index[close[0]] not in found:
                found.append(node_index[close[0]])

    return found[:max_entities]

link_entities("What do we know about " + hub + "?")

['Japan', 'bout', 'Abou']

In [14]:
def retrieve_facts(question, per_entity=12):
    """Entity linking + traversal → a list of plain-text facts."""
    entities = link_entities(question)
    facts = []

    for e in entities:                        # each entity's neighborhood
        facts.extend(facts_about(e)[:per_entity])

    if len(entities) >= 2:                    # the multi-hop magic: connect them
        for a in entities:
            for b in entities:
                if a < b:
                    path = path_between(a, b)
                    if path:
                        facts.append("Connection: " + "  →  ".join(path))

    return entities, list(dict.fromkeys(facts))       # dedupe, keep order

ents, facts = retrieve_facts(f"How is {hub} connected to the rest of the graph?")
print("Linked entities:", ents)
print(f"\n{len(facts)} facts retrieved; first few:")
for f in facts[:6]:
    print("  ", f)

Linked entities: ['the rest of', 'Connected', 'The rest']

13 facts retrieved; first few:
   Europe —has part→ the rest of
   Connected —publication date→ November 2002
   The rest —instance of→ escaped
   The rest —subclass of→ control
   The rest —part of→ history
   The rest —instance of→ captivity


In [15]:
# --- [4] Generation: pluggable LLM ----------------------------------------------------
# Check https://docs.claude.com for current model names if the one below has
# been superseded.
USE_LLM = bool(os.environ.get("ANTHROPIC_API_KEY"))

def generate_answer(question, facts):
    if not facts:
        return "I couldn't find anything in the graph about that."

    if not USE_LLM:
        # Template fallback: honest and grounded, just not fluent.
        return ("Here is what the knowledge graph contains:\n  - "
                + "\n  - ".join(facts))

    from anthropic import Anthropic
    client = Anthropic()
    # The prompt encodes RAG's core contract: compose ONLY from given facts.
    prompt = (
        "Answer the question using ONLY the facts below. Facts are written as "
        "'head —relation→ tail' triples from a knowledge graph.\n"
        "If the facts do not contain the answer, say exactly: "
        "'The graph does not contain that information.' Do not use outside "
        "knowledge. Cite which fact(s) you used.\n\n"
        "FACTS:\n" + "\n".join(f"- {f}" for f in facts)
        + f"\n\nQUESTION: {question}"
    )
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=400,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text

def ask(question):
    """The whole chatbot in one function."""
    entities, facts = retrieve_facts(question)
    answer = generate_answer(question, facts)
    print(f"Q: {question}")
    print(f"   linked entities: {entities}")
    print(f"A: {answer}\n")

print("LLM generation:", "ON (Claude API)" if USE_LLM else "OFF (template fallback)")

LLM generation: ON (Claude API)


Now we can talk to our graph!
Adapt these to whatever entities YOUR graph contains (see `hub` above, or the
Part 3 PageRank list).

In [16]:
ask(f"What do we know about {hub}?")
ask("How is Marie Curie connected to Frederic Joliot-Curie?")    # multi-hop!
ask("What is the capital of Australia?")    # NOT in the graph — watch it decline

Q: What do we know about Japan?
   linked entities: ['Japan', 'bout', 'Abou']
A: Based on the facts provided, here is what we know about Japan:

1. **Borders**: Japan shares a border with **China**, the **US**, and **Chinese**. *(Japan —shares border with→ China/US/Chinese)*

2. **Diplomatic Relations**: Japan has diplomatic relations with **China**, the **US**, **Finland**, **Chinese**, and **Europe**. *(Japan —diplomatic relation→ ...)*

3. **Body of Water**: Japan is located in or next to the **Sea of Japan**. *(Japan —located in or next to body of water→ Sea of Japan)*

4. **Classification**: Japan is an **instance of a country**. *(Japan —instance of→ country)*

5. **Language**: Japan uses the **Japanese language**. *(Japan —language used→ Japanese language)*

6. **Continent**: Japan is associated with **Europe**. *(Japan —continent→ Europe)*

7. **Sport**: Japan is connected to **wrestling**, which is a subclass of **bout**. *(Connection: Japan —sport— wrestling → wrestling —subc

### Grounded vs. hallucinated — the whole point, in one contrast

Look at the last question: **the system declines**. A bare LLM would cheerfully answer "Canberra" — correct here, but produced by exactly the mechanism that yields confident *wrong* answers elsewhere. Our chatbot instead reports that its knowledge (the graph) doesn't contain the answer.

That's the honest trade of RAG: **you exchange coverage for verifiability.** The system answers fewer questions, but every answer audits back to a specific edge — and that edge back to the sentence that produced it. For agent memory, that auditability is the feature, not a limitation.

And the multi-hop question was answered by `path_between()` — a traversal composing facts across sentences. Part 1's similarity baseline couldn't even *retrieve* the right chunks for questions like this. Time to make that comparison systematic.

---
## 4.2 Evaluation: how do we know it works?

"It answered my demo question" is not evaluation. RAG systems are evaluated on two independent axes — a failure on either sinks the answer:

| Axis | Question it asks | Failure mode |
|---|---|---|
| **Retrieval quality** | Did we fetch the facts needed to answer? | Right answer impossible: garbage in |
| **Answer quality** | Given the facts, is the answer faithful & relevant? | Right facts, wrong composition — or the model ignores facts and hallucinates |

Splitting them matters diagnostically: wrong answer + correct retrieval = a *generation* problem (prompt, model); wrong answer + bad retrieval = an *indexing* problem (extraction, linking, graph).

Below: a small **multi-hop question set**, comparing graph retrieval against the same TF-IDF baseline from Part 1 — both operating over the *same resolved text*, so it's a fair fight. Workshop-scale and qualitative, but the exact *shape* of a real evaluation.

In [17]:
# --- The baseline retriever (as in Part 1), over resolved sentences --------------------
class ChunkRetriever:
    def __init__(self, chunks):
        self.chunks = chunks
        self.vectorizer = TfidfVectorizer(stop_words="english")
        self.matrix = self.vectorizer.fit_transform(chunks)

    def retrieve(self, question, k=2):
        q_vec = self.vectorizer.transform([question])
        scores = cosine_similarity(q_vec, self.matrix)[0]
        ranked = scores.argsort()[::-1][:k]
        return [(self.chunks[i], float(scores[i])) for i in ranked]

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

# Part 2 caches one sentence per line in resolved_sentences.txt — use that
# instead of re-segmenting the full resolved text (which can be hundreds of MB).
SENT_FILE = CKPT / "resolved_sentences.txt"

if SENT_FILE.exists():
    print(f"[checkpoint] loaded {SENT_FILE}")
    resolved_sents = [s.strip() for s in SENT_FILE.read_text(encoding="utf-8").splitlines()
                      if len(s.strip()) > 10]
elif len(resolved_text) <= nlp.max_length:
    resolved_sents = [s.text.strip() for s in nlp(resolved_text).sents
                      if len(s.text.strip()) > 10]
else:
    def chunk_text(text, max_chars):
        """Split at sentence-ish boundaries so spaCy stays under max_length."""
        chunks, start = [], 0
        while start < len(text):
            end = min(start + max_chars, len(text))
            if end < len(text):
                for sep in (" . ", ". ", " "):
                    cut = text.rfind(sep, start, end)
                    if cut > start:
                        end = cut + len(sep)
                        break
            chunks.append(text[start:end])
            start = end
        return chunks

    resolved_sents = []
    with nlp.select_pipes(disable=["ner"]):
        for chunk_doc in nlp.pipe(chunk_text(resolved_text, 100_000), batch_size=8):
            for s in chunk_doc.sents:
                sent = " ".join(s.text.split())
                if len(sent) > 10:
                    resolved_sents.append(sent)

baseline = ChunkRetriever(resolved_sents)
print(f"Baseline indexes {len(resolved_sents):,} sentences")

[checkpoint] loaded checkpoints/resolved_sentences.txt
Baseline indexes 3,692,621 sentences


In [18]:
# --- A tiny multi-hop evaluation set ----------------------------------------------------
# Each item names the entities the question is REALLY about — our retrieval
# gold labels. If you built the graph from a live wikitext slice, REWRITE these
# for your entities: evaluation sets are always corpus-specific.
eval_set = [
    {"q": "What field did Marie Curie's husband work in?",
     "gold_entities": ["Marie Curie", "Pierre Curie"]},
    {"q": "How is Marie Curie connected to Frederic Joliot-Curie?",
     "gold_entities": ["Marie Curie", "Frederic Joliot-Curie"]},
    {"q": "Where was Marie Curie born?",
     "gold_entities": ["Marie Curie", "Warsaw"]},
]

def graph_retrieval_hits(item):
    """Did graph retrieval surface facts touching every gold entity?"""
    _, facts = retrieve_facts(item["q"])
    blob = " ".join(facts).lower()
    return [e for e in item["gold_entities"] if e.lower() in blob]

def baseline_retrieval_hits(item, k=2):
    """Did the top-k similar chunks mention every gold entity?"""
    blob = " ".join(chunk for chunk, _ in baseline.retrieve(item["q"], k=k)).lower()
    return [e for e in item["gold_entities"] if e.lower() in blob]

print(f"{'question':<55} {'graph':>8} {'baseline':>10}")
for item in eval_set:
    g, b = graph_retrieval_hits(item), baseline_retrieval_hits(item)
    n = len(item["gold_entities"])
    print(f"{item['q'][:53]:<55} {len(g)}/{n:>6} {len(b)}/{n:>8}")

# Reading the results honestly: the graph covers all gold entities because it
# FOLLOWS edges. On a tiny, fully coref-resolved corpus the baseline may also
# pass — every sentence now names entities explicitly, so word overlap works
# better than it would on raw text. (Notice: coref helped the BASELINE too.)
# The gap opens as the corpus grows and connecting facts drift far apart in
# the text: similarity can't follow a chain of mentions, traversal can.
# Try it: scale up target_chars in Part 1 and add questions whose hops span
# distant paragraphs.

question                                                   graph   baseline
What field did Marie Curie's husband work in?           2/     2 2/       2
How is Marie Curie connected to Frederic Joliot-Curie   1/     2 1/       2
Where was Marie Curie born?                             1/     2 1/       2


In [19]:
# --- A crude faithfulness (groundedness) check --------------------------------------
# Faithfulness: does the ANSWER stay inside the retrieved FACTS? Real evaluators
# use an LLM-as-judge to verify each claim; here is the idea in miniature —
# flag answer entities that never appear in the retrieved facts.
def unsupported_entities(question):
    entities, facts = retrieve_facts(question)
    answer = generate_answer(question, facts)
    facts_blob = " ".join(facts).lower()
    answer_ents = {ent.text for ent in nlp(answer).ents}
    unsupported = [e for e in answer_ents if e.lower() not in facts_blob]
    return answer, unsupported

answer, unsupported = unsupported_entities(f"What do we know about {hub}?")
print(answer[:300], "…" if len(answer) > 300 else "")
print("\nEntities in answer but NOT in retrieved facts:", unsupported or "none ✓")

Based on the facts provided, here is what we know about Japan:

1. **Borders**: Japan shares a border with **China**, the **US**, and **Chinese**. *(Facts: "Japan —shares border with→ China", "Japan —shares border with→ US", "Japan —shares border with→ Chinese")*

2. **Diplomatic Relations**: Japan  …

Entities in answer but NOT in retrieved facts: ['4', '2', '6', '3', '1', '5']


<center>
     <img src="data/D4Sci_logo_full.png" alt="Data For Science, Inc" align="center" border="0" width=300px> 
</center>